# Project 2 - NCSU ST 554
### Author: Trevor Lillywhite
### Due Date: March 24, 2026

## Part I - Creating a Class

**Objective:** Create our own class called `SparkDataCheck` that works on Spark SQL style data frames. 

This will be created in a `.py` file and tested here in a pySpark3 notebook.

The `SparkDataCheck` class can be used to perform various checks on a spark DataFrame. 

*Instantiation:*
+ The default instantiation is to create an object by directly supplying a spark dataframe. 
+ There are also two class methods to alternatively instantiate an object using a csv file (`.from_csv()`) or a `pandas` DataFrame (`.from_pandas()`).

*Validation Methods:* There are three validation methods.
+ `.check_within_limits()` checks if each value in a numeric column is within user defined limits (inclusive). It returns a dataframe with an appended column of Boolean values.
+ `.check_string()` checks if each value in a string column falls within a user specified set of levels and returns the dataframe with an appended column of Boolean values.
+ `.check_Null()` checks if each value in a column is missing (`NULL`) and returns a dataframe with an appended column of Boolean values.

*Summarization Methods:* There are two summarization methods.
+ `.find_minmax()` reports the min and max of a numeric column specified by the user as a column name. There is an optional grouping variable. If no grouping variable is supplied, the min and max are across the entire column. If a grouping variable is supplied, the min and max of each group is reported. If no column is specified, then all numeric columns are reported (including grouping, if specified). 
+ `.find_counts` reports the counts associated with one or two string columns. The first string column is required and the second is optional. 

Below, the class will be tested to verify functionality. The class, which was defined in a `.py` file, will be imported in this notebook for testing. 

#### Import script

This will give access to the class I created

In [2]:
# Temporarily change working directory
%cd Project_2          # Note: if directory is already changed, this will return a non-fatal error

# Import relevant modules
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql import functions as F
import pyspark.pandas as ps

# Import script
import project2_script

[Errno 2] No such file or directory: 'Project_2 # Note: if directory is already changed, this will return a non-fatal error'
/home/jupyter-tflillyw@ncsu.edu/Project_2


In [3]:
# Add cell to reload module (supports troubleshooting)
import importlib
importlib.reload(project2_script)

<module 'project2_script' from '/home/jupyter-tflillyw@ncsu.edu/Project_2/project2_script.py'>

result_via_pandas = project2_script.SparkDataCheck.from_pandas(test_session, p_df)
result_via_pandas

result_via_pandas.df.show()

result_via_csv = project2_script.SparkDataCheck.from_csv(test_session, 'weekly_nfl_data.csv')
result_via_csv

result_via_csv.check_within_limits('position', upper=1)

check3 = result_via_csv.check_Null('player_name')
check3.df.select('player_name', 'player_name_is_Null').orderBy(F.rand()).show(10)

check4 = result_via_csv.find_minmax(column = 'rushing_yards', group = 'season')
check4

check5 = result_via_csv.find_minmax(column='season')
check5

check6 = result_via_csv.find_counts(column1 = 'position', 
                                    column2 = 'opponent_team')
check6

In [ ]:
#   END TROUBLESHOOTING

### Read in air quality data

The air quality dataset will be used to test the class's functionality. 

This was used in the first project and is available here: https://www4.stat.ncsu.edu/online/datasets/air.csv

We will use the custom-defined method to create an instance of the class from the csv file downloaded from the website and uploaded to the JupyterHub environment in the same folder as the `.py` script and this notebook.

To do that, we first need to create a spark session.

In [5]:
# Create test session for the air quality data
air_session = SparkSession.builder.appName('my_app')\
                    .config("spark.sql.ansi.enabled", "false")\
                    .getOrCreate()

Now, we can create our class instance.

In [7]:
air_data = project2_script.SparkDataCheck.from_csv(
    air_session, 'air.csv')
air_data.df.show(1)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
only showing top 1 row


26/03/22 20:18:22 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-tflillyw@ncsu.edu/Project_2/air.csv


There is a warning above because the csv file didn't include a name for the first column. The program was still able to run, as it created a name for that column (`_c0`). 

Before we test our various methods on the class instance, we need to do a small amount of data cleaning. First, we will clean the column names so they don't include periods, which could be misinterpreted. Additionally, any missing values are encoded as the value "-200", which will skew our numerical statistics. Further, since Date is the only string column, we will cast the Time column to be a string so we can better test the associated method. We will also name the first column of the csv to prevent the warning from displaying again. 

In [8]:
# Clean dataset column names

# Rename first column (was unnamed in csv)
air_data.df = air_data.df.withColumnRenamed('_c0', 'observation')

# Rename other columns (remove decimals)
air_data.df = air_data.df.withColumnRenamed('PT08.S1(CO)', 'PT08_S1(CO)')
air_data.df = air_data.df.withColumnRenamed('PT08.S2(NMHC)', 'PT08_S2(NMHC)')
air_data.df = air_data.df.withColumnRenamed('PT08.S3(NOx)', 'PT08_S3(NOx)')
air_data.df = air_data.df.withColumnRenamed('PT08.S4(NO2)', 'PT08_S4(NO2)')
air_data.df = air_data.df.withColumnRenamed('PT08.S5(O3)', 'PT08_S5(O3)')

# Write to CSV to permanently change header names
air_data.df.write.csv("air_modified.csv", header=True, mode="overwrite")

26/03/22 20:18:30 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-tflillyw@ncsu.edu/Project_2/air.csv


In [9]:
# Read modified csv
air_data = project2_script.SparkDataCheck.from_csv(
    air_session, 'air_modified.csv')
air_data.df.show(1)

+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|observation|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|
+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|          0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
only showing top 1 row


In [10]:
air_data.df.describe().show()

26/03/22 20:18:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+--------+------------------+------------------+-------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+------------------+-----------------+-----------------+------------------+
|summary|       observation|    Date|            CO(GT)|       PT08_S1(CO)|           NMHC(GT)|          C6H6(GT)|    PT08_S2(NMHC)|          NOx(GT)|      PT08_S3(NOx)|           NO2(GT)|      PT08_S4(NO2)|       PT08_S5(O3)|                T|               RH|                AH|
+-------+------------------+--------+------------------+------------------+-------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+------------------+-----------------+-----------------+------------------+
|  count|              9357|    9357|              9357|              9357|               9357|              9357|             9357|             9357|    

In [11]:
# Clean dataset values
# Replace values of "-200" with `NULL`. 
air_data.df = air_data.df.replace(-200, None)
air_data.df.describe().show()

+-------+------------------+--------+------------------+------------------+------------------+-----------------+------------------+-----------------+-----------------+------------------+------------------+------------------+-----------------+------------------+-----------------+
|summary|       observation|    Date|            CO(GT)|       PT08_S1(CO)|          NMHC(GT)|         C6H6(GT)|     PT08_S2(NMHC)|          NOx(GT)|     PT08_S3(NOx)|           NO2(GT)|      PT08_S4(NO2)|       PT08_S5(O3)|                T|                RH|               AH|
+-------+------------------+--------+------------------+------------------+------------------+-----------------+------------------+-----------------+-----------------+------------------+------------------+------------------+-----------------+------------------+-----------------+
|  count|              9357|    9357|              7674|              8991|               914|             8991|              8991|             7718|           

We successfully eliminated the -200 encoding of missing values. The only cleaning step remaining is to change the Time column to be a string.

In [12]:
# Cast Date and Time columns to be strings
air_data.df = air_data.df.withColumn("Time", F.col("Time").cast("string"))
air_data.df.printSchema()

root
 |-- observation: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08_S1(CO): integer (nullable = true)
 |-- NMHC(GT): integer (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08_S2(NMHC): integer (nullable = true)
 |-- NOx(GT): integer (nullable = true)
 |-- PT08_S3(NOx): integer (nullable = true)
 |-- NO2(GT): integer (nullable = true)
 |-- PT08_S4(NO2): integer (nullable = true)
 |-- PT08_S5(O3): integer (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)



We are finally ready to test our class!

### Provide examples using each method

We will provide a few examples of using each of the methods on this object. We will do this robustly to show that error messages show when appropriate and when the method can run using varying amounts of arguments. 

In some cases, we will randomize the values returned, effectively shuffling the deck to let us see a broad cross-seciton of data (not limited by the behavior of the first several observations). This randomization will change every time the code is run.

#### Test `check_within_limits()` method

**Test 1:** Provide no limits (bounds).

**Expected behavior:** Error

In [13]:
test1 = air_data.check_within_limits('T')

Error: Must specify at least one upper or lower bound.


*Result:* Success!

**Test 2:** Provide both limits (bounds). Provide non-numeric column.

**Expected behavior:** Warning, but still return unmodified dataframe.

In [14]:
test2 = air_data.check_within_limits('Date', lower=0, upper=1000)
test2.df.orderBy(F.rand()).show(10)

+-----------+----------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|observation|      Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|
+-----------+----------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|        683|  4/8/2004|2026-03-22 05:00:00|   0.8|        869|      68|     2.8|          642|     69|        1291|     62|        1288|        723| 8.1|72.1|0.7798|
|       6955|12/25/2004|2026-03-22 13:00:00|   5.5|       1609|    NULL|    18.1|         1243|    602|         505|    178|        1491|       2100|11.4|62.9|0.8454|
|       2724|  7/2/2004|2026-03-22 06:00:00|   0.8|        993|    NULL|     5.6|          795|     60|         888|     63|        1629|        770|24.8|50.6|1.5654

*Result:* Success!

**Test 3:** Provide both limits (bounds). Provide numeric column.

**Expected behavior:** Successful execution. New column with Boolean results.

In [15]:
test3 = air_data.check_within_limits('T', lower=20, upper=30)
test3.df.orderBy(F.rand()).show(10)

+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|observation|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|T_within_limits|
+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|       2104| 6/6/2004|2026-03-22 10:00:00|   0.9|        806|    NULL|     3.4|          675|     61|        1270|     61|        1414|        528|23.1|39.4|1.0983|           true|
|       2093| 6/5/2004|2026-03-22 23:00:00|   1.7|       1008|    NULL|     8.0|          900|     55|         907|     67|        1681|        850|21.6|53.6|1.3674|           true|
|        370|3/26/2004|2026-03-22 04:00:00|   0.7|        925|      40|     2.2|          

*Result:* Success!

**Test 4:** Provide lower limit only. Provide numeric column.

**Expected behavior:** Successful execution. New column with Boolean results. Range is interpreted as anything above the lower limit.

In [16]:
test4 = air_data.check_within_limits('T', lower=20)
test4.df.orderBy(F.rand()).show(10)

+-----------+----------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|observation|      Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|T_within_limits|
+-----------+----------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|       8244| 2/17/2005|2026-03-22 06:00:00|   0.5|        893|    NULL|     1.1|          507|     59|        1111|     51|         794|        489| 6.9|42.2|0.4224|          false|
|       7959|  2/5/2005|2026-03-22 09:00:00|   4.1|       1165|    NULL|    11.3|         1024|    759|         646|    310|        1134|       1365| 4.1|52.7|0.4345|          false|
|       8738|  3/9/2005|2026-03-22 20:00:00|   5.9|       1552|    NULL|    24.4|    

*Result:* Success!

**Test 5:** Provide upper limit only. Provide numeric column.

**Expected behavior:** Successful execution. New column with Boolean results. Range is interpreted as anything below the upper limit.

In [19]:
test5 = air_data.check_within_limits('T', upper=22)
test5.df.orderBy(F.rand()).show(10)

+-----------+----------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|observation|      Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|T_within_limits|
+-----------+----------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|       6928|12/24/2004|2026-03-22 10:00:00|   6.6|       1462|    NULL|    18.8|         1263|    750|         510|    170|        1421|       2082| 6.7|59.7| 0.589|           true|
|       7247|  1/6/2005|2026-03-22 17:00:00|   1.9|       1173|    NULL|     7.7|          885|    274|         685|    134|        1288|       1038|15.6|54.8| 0.966|           true|
|       4151| 8/30/2004|2026-03-22 17:00:00|  NULL|       1099|    NULL|    11.9|    

*Result:* Success!

#### Test `check_string()` method

**Test 6:** Provide invalid column name (not in DataFrame).

**Expected behavior:** Error.

In [20]:
test6 = air_data.check_string('bad_name', levels='wrong')

Error: Invalid column (not in DataFrame)


*Result:* Success!

**Test 7:** Provide numeric column.

**Expected behavior:** Error.

In [21]:
test7 = air_data.check_string('T', levels='25')

*Result:* Success!

**Test 8:** Provide string column. Provide single "level".

**Expected behavior:** Successful execution. New column with Boolean results. (Note: no result randomization this time)

In [22]:
test8 = air_data.check_string('Date', levels='3/10/2004')
test8.df.show(10)

+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+
|observation|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|T_within_limits|Date_value_in_list|
+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+
|          0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|           true|              true|
|          1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|           true|              t

*Result:* Success!

**Test 9:** Provide string column. Provide multiple "levels".

**Expected behavior:** Successful execution. New column with Boolean results. (Note: no result randomization this time)

In [23]:
test9 = air_data.check_string('Date', levels=['3/10/2004', '3/12/2004'])
test9.df.show(35)

+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+
|observation|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|T_within_limits|Date_value_in_list|
+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+
|          0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|           true|              true|
|          1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|           true|              t

*Result:* Success!

**Test 10:** Provide string column. Provide empty list of "levels".

**Expected behavior:** Successful execution. New column with Boolean results, all "false". (Note: no result randomization this time)

In [24]:
test10 = air_data.check_string('Date', levels=[])
test10.df.show(10)

+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+
|observation|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|T_within_limits|Date_value_in_list|
+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+
|          0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|           true|             false|
|          1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|           true|             fa

*Result:* Success!

#### Test `check_Null()` method

**Test 11:** Provide invalid column name (not in DataFrame).

**Expected behavior:** Error.

In [25]:
test11 = air_data.check_Null('bad_name')

Error: Invalid column (not in DataFrame)


*Result:* Success!

**Test 12:** Provide column name that isn't a string.

**Expected behavior:** Error.

In [26]:
test12 = air_data.check_Null(column=12)

Error: Column name must be a string


*Result:* Success!

**Test 13:** Provide numeric column.

**Expected behavior:** Successful execution. New column with Boolean results. (Note: might need to be run several times to actually find a NULL result)

In [27]:
test13 = air_data.check_Null('NOx(GT)')
test13.df.show(25)

+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+---------------+
|observation|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|T_within_limits|Date_value_in_list|NOx(GT)_is_Null|
+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+---------------+
|          0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|           true|             false|          false|
|          1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1

*Result:* Success!

**Test 14:** Provide string column.

**Expected behavior:** Successful execution. New column with Boolean results. (Note: Date and Time are never null, so this shouldn't ever produce a "true" Boolean result.)

In [28]:
test14 = air_data.check_Null('Date')
test14.df.show(35)

+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+---------------+------------+
|observation|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|T_within_limits|Date_value_in_list|NOx(GT)_is_Null|Date_is_Null|
+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+---------------+------------+
|          0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|           true|             false|          false|       false|
|          1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|

*Result:* Success!

We also see the method "chaining" even though we aren't directly chaining in the same line of code because each time we call it, we modify the underlying dataframe inside the object. Similar results would be observed if we directly chained the methods in a single line of code.

#### Test `find_minmax()` method

**Test 15:** Provide bad column name (not in DataFrame)

**Expected behavior:** Error

In [29]:
test15 = air_data.find_minmax(column='football')
test15

Error: Invalid column (not in DataFrame)


*Result:* Success!

**Test 16:** Provide non-numeric column.

**Expected behavior:** Error

In [30]:
test16 = air_data.find_minmax('Date')
test16

Error: Column Date is non-numeric. Provide a numeric column.


*Result:* Success!

**Test 17:** Provide numeric column. Do not provide a group.

**Expected behavior:** Successful execution. Results will be a `pandas` DataFrame simply reporting the min and max values for that column.

In [31]:
test17 = air_data.find_minmax(column='NOx(GT)')
test17

,Min,Max
NOx(GT),2,1479


*Result:* Success!

**Test 18:** Provide numeric column. Provide a group.

**Expected behavior:** Successful execution. Results will be a `pandas` DataFrame reporting the min and max values for that column grouped by the 'group' argument.

In [32]:
test18 = air_data.find_minmax(column='NOx(GT)', group='Date')
test18

,NOx(GT)_min,NOx(GT)_max
Date,,
1/1/2005,145.0,832.0
1/10/2005,64.0,698.0
1/11/2005,96.0,662.0
1/12/2005,162.0,910.0
1/13/2005,161.0,760.0
...,...,...
9/5/2004,NaN,NaN
9/6/2004,NaN,NaN
9/7/2004,145.0,232.0


*Result:* Success!

**Test 19:** Do not provide a column or group.

**Expected behavior:** Successful execution. Results will be a `pandas` DataFrame reporting the min and max values for each numeric column with no grouping.

In [33]:
test19 = air_data.find_minmax()
test19

,min,max
observation,0.0000,9356.000
CO(GT),0.1000,11.900
PT08_S1(CO),647.0000,2040.000
NMHC(GT),7.0000,1189.000
C6H6(GT),0.1000,63.700
PT08_S2(NMHC),383.0000,2214.000
NOx(GT),2.0000,1479.000
PT08_S3(NOx),322.0000,2683.000
NO2(GT),2.0000,340.000
PT08_S4(NO2),551.0000,2775.000


*Result:* Success!

**Test 20:** Provide a group. Do not provide a column.

**Expected behavior:** Successful execution. Results will be a `pandas` DataFrame reporting the min and max values for each numeric column, grouped by the 'group' argument.

In [34]:
test20 = air_data.find_minmax(group='Date')
test20

,observation_min,observation_max,CO(GT)_min,CO(GT)_max,PT08_S1(CO)_min,PT08_S1(CO)_max,NMHC(GT)_min,NMHC(GT)_max,C6H6(GT)_min,C6H6(GT)_max,...,PT08_S4(NO2)_min,PT08_S4(NO2)_max,PT08_S5(O3)_min,PT08_S5(O3)_max,T_min,T_max,RH_min,RH_max,AH_min,AH_max
0,,,,,,,,,,,,,,,,,,,,,
1/1/2005,7110,7133,1.0,4.7,915.0,1472.0,NaN,NaN,3.0,16.6,...,897.0,1344.0,879.0,1735.0,2.6,12.8,32.3,63.2,0.4375,0.5961
1/10/2005,7326,7349,0.4,3.5,958.0,1523.0,NaN,NaN,1.3,21.4,...,1122.0,1804.0,707.0,1634.0,12.3,14.4,63.6,73.3,0.9912,1.0567
1/11/2005,7350,7373,0.7,5.1,977.0,1526.0,NaN,NaN,2.1,26.5,...,1106.0,1855.0,830.0,1738.0,11.7,13.8,56.3,71.9,0.8754,1.0087
1/12/2005,7374,7397,1.0,5.9,1002.0,1526.0,NaN,NaN,4.2,28.4,...,1139.0,1902.0,958.0,1770.0,8.7,15.8,48.8,79.4,0.8428,0.9467
1/13/2005,7398,7421,0.9,5.2,891.0,1523.0,NaN,NaN,3.2,26.6,...,1054.0,1811.0,798.0,1783.0,6.2,14.2,54.1,82.1,0.7751,0.9166
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9/5/2004,4278,4301,0.6,5.8,784.0,1399.0,NaN,NaN,1.9,30.3,...,1060.0,2052.0,342.0,1652.0,25.4,34.5,17.5,38.6,0.8162,1.2477
9/6/2004,4302,4325,0.4,2.9,826.0,1133.0,NaN,NaN,1.0,13.5,...,1189.0,1669.0,385.0,1105.0,23.5,30.1,26.7,49.6,1.1217,1.4247
9/7/2004,4326,4349,0.6,4.1,847.0,1311.0,NaN,NaN,2.7,22.0,...,1258.0,1957.0,497.0,1486.0,21.9,32.5,19.4,48.3,0.9279,1.3688


*Result:* Success!

#### Test `find_counts()` method

**Test 21:** Provide a bad column name (not in the DataFrame).

**Expected behavior:** Error.

In [35]:
test21 = air_data.find_counts(column1='date')
test21

Error: Invalid column (not in DataFrame)


*Result:* Success!

**Test 22:** Provide a single column name for a numeric column.

**Expected behavior:** Error.

In [36]:
test22 = air_data.find_counts(column1='T')
test22

Error: Column T is numeric. Provide a string column.


*Result:* Success!

**Test 23:** Provide a numeric column and then a string column.

**Expected behavior:** Error.

In [37]:
test23 = air_data.find_counts(column1='RH', column2='Date')
test23

Error: Column RH is numeric. Provide a string column.


*Result:* Success!

**Test 24:** Provide a single string column.

**Expected behavior:** Successful execution. Results will be a `pandas` DataFrame reporting the counts for each level of that column's values.

In [38]:
test24 = air_data.find_counts(column1='Date')
test24

,Date,Date_count
0,9/2/2004,24
1,12/26/2004,24
2,2/18/2005,24
3,10/10/2004,24
4,10/11/2004,24
...,...,...
386,8/16/2004,24
387,12/20/2004,24
388,3/5/2005,24
389,4/4/2005,15


*Result:* Success!

**Test 25:** Provide two string columns.

**Expected behavior:** Successful execution. Results will be a pandas DataFrame reporting the counts for each level of each column's values.

In [39]:
test25 = air_data.find_counts(column1='Date', column2='Time')
test25

,Date,Date_count,Time,Time_count
0,9/2/2004,24,2026-03-22 23:00:00,390.0
1,12/26/2004,24,2026-03-22 06:00:00,390.0
2,2/18/2005,24,2026-03-22 00:00:00,390.0
3,10/10/2004,24,2026-03-22 20:00:00,390.0
4,10/11/2004,24,2026-03-22 22:00:00,390.0
...,...,...,...,...
386,8/16/2004,24,NaN,NaN
387,12/20/2004,24,NaN,NaN
388,3/5/2005,24,NaN,NaN
389,4/4/2005,15,NaN,NaN


*Result:* Success!

**Test 26:** Chain two validation methods. We will use untouched columns to prove that previous methods/results didn't affect this. 

**Expected behavior:** Successful execution. New Boolean columns will be added for 'NMHC(GT)_is_Null' and 'PT08_S1(CO)_within_limits'. 

In [40]:
test26 = air_data.check_Null('NMHC(GT)').check_within_limits('PT08_S1(CO)', lower=900, upper=1300)
test26.df.orderBy(F.rand()).show(25)

+-----------+----------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+---------------+------------+----------------+-------------------------+
|observation|      Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|T_within_limits|Date_value_in_list|NOx(GT)_is_Null|Date_is_Null|NMHC(GT)_is_Null|PT08_S1(CO)_within_limits|
+-----------+----------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+---------------+------------+----------------+-------------------------+
|       8422| 2/24/2005|2026-03-22 16:00:00|   3.0|       1274|    NULL|    13.3|         1094|    507|         537|    267|        1364|       1428| 7.6|70.8|0.7399|           t

*Result:* Success!

We can chain validation methods and also execute any validation or summarization method.

### Read in same data set using `pandas`

(Note that this is _not_ done using `pandas-on-spark`)

First, we will create a `pandas` DataFrame using the csv file. Then we will execute similar data cleaning methods to those used with the `from_csv()` method before executing our test.

In [41]:
# Create pandas DataFrame from CSV
air_panda = pd.read_csv('air.csv')
air_panda

,Unnamed: 0,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9352,9352,4/4/2005,10:00:00,3.1,1314,-200,13.5,1101,472,539,190,1374,1729,21.9,29.3,0.7568
9353,9353,4/4/2005,11:00:00,2.4,1163,-200,11.4,1027,353,604,179,1264,1269,24.3,23.7,0.7119
9354,9354,4/4/2005,12:00:00,2.4,1142,-200,12.4,1063,293,603,175,1241,1092,26.9,18.3,0.6406
9355,9355,4/4/2005,13:00:00,2.1,1003,-200,9.5,961,235,702,156,1041,770,28.3,13.5,0.5139


In [42]:
# Rename columns for convenience
air_panda = air_panda.rename(columns={'Unnamed: 0': 'observation',
                                      'PT08.S1(CO)': 'PT08_S1(CO)',
                                      'PT08.S2(NMHC)': 'PT08_S2(NMHC)',
                                      'PT08.S3(NOx)': 'PT08_S3(NOx)',
                                      'PT08.S4(NO2)': 'PT08_S4(NO2)',
                                      'PT08.S5(O3)': 'PT08_S5(O3)'})

# Change Time column to be string-type
air_panda['Time'] = air_panda['Time'].astype('str')

# Replace missing values of -200 with NaN
import numpy as np
air_panda = air_panda.replace(-200, np.NaN)

air_panda.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9357 entries, 0 to 9356
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   observation    9357 non-null   int64  
 1   Date           9357 non-null   object 
 2   Time           9357 non-null   object 
 3   CO(GT)         7674 non-null   float64
 4   PT08_S1(CO)    8991 non-null   float64
 5   NMHC(GT)       914 non-null    float64
 6   C6H6(GT)       8991 non-null   float64
 7   PT08_S2(NMHC)  8991 non-null   float64
 8   NOx(GT)        7718 non-null   float64
 9   PT08_S3(NOx)   8991 non-null   float64
 10  NO2(GT)        7715 non-null   float64
 11  PT08_S4(NO2)   8991 non-null   float64
 12  PT08_S5(O3)    8991 non-null   float64
 13  T              8991 non-null   float64
 14  RH             8991 non-null   float64
 15  AH             8991 non-null   float64
dtypes: float64(13), int64(1), object(2)
memory usage: 1.1+ MB


In [43]:
air_panda.describe()

,observation,CO(GT),PT08_S1(CO),NMHC(GT),C6H6(GT),PT08_S2(NMHC),NOx(GT),PT08_S3(NOx),NO2(GT),PT08_S4(NO2),PT08_S5(O3),T,RH,AH
count,9357.000000,7674.000000,8991.000000,914.000000,8991.000000,8991.000000,7718.000000,8991.000000,7715.000000,8991.000000,8991.000000,8991.000000,8991.000000,8991.000000
mean,4678.000000,2.152750,1099.833166,218.811816,10.083105,939.153376,246.896735,835.493605,113.091251,1456.264598,1022.906128,18.317829,49.234201,1.025530
std,2701.277568,1.453252,217.080037,204.459921,7.449820,266.831429,212.979168,256.817320,48.370108,346.206794,398.484288,8.832116,17.316892,0.403813
min,0.000000,0.100000,647.000000,7.000000,0.100000,383.000000,2.000000,322.000000,2.000000,551.000000,221.000000,-1.900000,9.200000,0.184700
25%,2339.000000,1.100000,937.000000,67.000000,4.400000,734.500000,98.000000,658.000000,78.000000,1227.000000,731.500000,11.800000,35.800000,0.736800
50%,4678.000000,1.800000,1063.000000,150.000000,8.200000,909.000000,180.000000,806.000000,109.000000,1463.000000,963.000000,17.800000,49.600000,0.995400
75%,7017.000000,2.900000,1231.000000,297.000000,14.000000,1116.000000,326.000000,969.500000,142.000000,1674.000000,1273.500000,24.400000,62.500000,1.313700
max,9356.000000,11.900000,2040.000000,1189.000000,63.700000,2214.000000,1479.000000,2683.000000,340.000000,2775.000000,2523.000000,44.600000,88.700000,2.231000


Now we will create another instance of the `SparkDataCheck` class starting with the `pandas` data frame instead of reading the csv directly into the instance. 

In [45]:
# Create another test session for the air quality data
air_session2 = SparkSession.builder.appName('my_app2')\
                    .config("spark.sql.ansi.enabled", "false")\
                    .getOrCreate()
air_data2 = project2_script.SparkDataCheck.from_pandas(air_session2, air_panda)
air_data2.df.show(1)

+-----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|observation|     Date|    Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|
+-----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|          0|3/10/2004|18:00:00|   2.6|     1360.0|   150.0|    11.9|       1046.0|  166.0|      1056.0|  113.0|      1692.0|     1268.0|13.6|48.9|0.7578|
+-----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
only showing top 1 row


In [46]:
air_data2.df.printSchema()

root
 |-- observation: long (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08_S1(CO): double (nullable = true)
 |-- NMHC(GT): double (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08_S2(NMHC): double (nullable = true)
 |-- NOx(GT): double (nullable = true)
 |-- PT08_S3(NOx): double (nullable = true)
 |-- NO2(GT): double (nullable = true)
 |-- PT08_S4(NO2): double (nullable = true)
 |-- PT08_S5(O3): double (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)



### Provide an example method call

Here, we will just show use of one method call since we previously demonstrated the ability to use each method. 

To maximize value, we will chain two validation methods together (similar to Test 26). 

**Test 27:** Chain two validation methods. 

**Expected behavior:** Successful execution. New Boolean columns will be added for 'NMHC(GT)_is_Null' and 'PT08_S1(CO)_within_limits'. 

In [47]:
test27 = air_data2.check_Null('NMHC(GT)').check_within_limits('PT08_S1(CO)', lower=900, upper=1300)
test27.df.orderBy(F.rand()).show(25)

+-----------+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----------------+-------------------------+
|observation|      Date|    Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|NMHC(GT)_is_Null|PT08_S1(CO)_within_limits|
+-----------+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----------------+-------------------------+
|        104| 3/15/2004| 2:00:00|   1.8|     1224.0|    66.0|     7.0|        855.0|  108.0|       998.0|   88.0|      1566.0|     1149.0|13.4|61.3|0.9361|           false|                     true|
|       1257|  5/2/2004| 3:00:00|   1.6|     1000.0|    NULL|     6.6|        838.0|   NULL|       910.0|   NULL|      1580.0|      942.0|14.8|67.6|1.1352|            true|                     true|
|    

*Results:* Success! The additional columns showed up, and the values make sense. 

This shows we can equivalently add data from a csv directly into the class, or from csv to `pandas` to the class. 

## Part II - Basic spark analysis on NFL data

In this part, we will use Spark SQL and `pandas-on-spark` to do some basic analysis on some NFL data.  The data was directly sourced from a csv file provided by the course instructor. 

The first section will be completed solely using `pandas-on-Spark`